In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

d:\mis\VSCode\Projects\twitter-sentiment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# CONFIG
# =========================
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
CONF_THRESHOLD = 0.8
MAX_LEN = 128
BATCH_SIZE = 16

LABEL_MAP = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

In [3]:
# =========================
# LOAD DATA
# =========================
df = pd.read_excel(r"data/reviews_cleaned.xlsx")
df = df.dropna(subset=["text"])

In [4]:
# =========================
# LOAD MODEL (for pseudo-labeling)
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13362.74it/s]


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [5]:
# =========================
# TEXT CLEANING (simple)
# =========================
def clean_text(text):
    text = text.lower()
    return text

df["text"] = df["text"].apply(clean_text)

In [6]:
# =========================
# PSEUDO-LABELING
# =========================
def predict_batch(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()

pseudo_labels = []
confidences = []

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_texts = df["text"].iloc[i:i+BATCH_SIZE].tolist()
    probs = predict_batch(batch_texts)

    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)

    pseudo_labels.extend(preds)
    confidences.extend(confs)

df["label"] = pseudo_labels
df["confidence"] = confidences

  0%|          | 0/422 [00:00<?, ?it/s]

 15%|█▍        | 63/422 [02:06<11:59,  2.01s/it]


KeyboardInterrupt: 

In [ ]:
# =========================
# FILTER HIGH-CONFIDENCE
# =========================
df_filtered = df[df["confidence"] >= CONF_THRESHOLD].copy()

print("Original:", len(df))
print("Filtered:", len(df_filtered))

In [ ]:
# =========================
# TRAIN / VALID SPLIT
# =========================
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_filtered,
    test_size=0.1,
    stratify=df_filtered["label"],
    random_state=42
)

In [ ]:
# =========================
# CONVERT TO HF DATASET
# =========================
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])

In [ ]:
# =========================
# TOKENIZATION
# =========================
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# =========================
# LOAD MODEL FOR TRAINING
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.to(device)

In [ ]:
# =========================
# TRAINING SETUP
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

In [ ]:
# =========================
# METRICS
# =========================
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
# =========================
# TRAIN
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# =========================
# SAVE MODEL
# =========================
trainer.save_model("./sentiment-model")
tokenizer.save_pretrained("./sentiment-model")

print("Model saved!")